
# ENDS-Shortlist Arena140K Complete Experiment Notebook

This notebook is the **non-toy, real-data** implementation of the ENDS-Shortlist pipeline for `lmarena-ai/arena-human-preference-140k`.

It does not create synthetic data and it does not use a toy default such as `K=8` or `budget=5`. The default configuration below is the main real-data experiment:

- `K_TARGET = 20`
- `MIN_PAIR_SUPPORT = 20`
- `BUDGETS = [500, 1000, 2000, 5000, 10000, 20000]`
- algorithms: ENDS-Shortlist, Uniform, Arena-frequency, Thompson-shortlist, Borda-LUCB-shortlist, Top1-ENDS-plus-padding, Empirical-top3-frequency
- optional fixed-confidence calibration/test grid

Before running, download the parquet files from:

https://huggingface.co/datasets/lmarena-ai/arena-human-preference-140k/tree/main

Then set `LOCAL_PARQUET_GLOB` to your local path.


In [ ]:

# Install once if needed, then restart the kernel.
# !pip install -r requirements_ends_shortlist_complete.txt


In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

from ends_shortlist_arena import (
    load_arena_dataframe, filter_arena_dataframe, greedy_select_models, build_graph,
    build_pair_reservoirs, split_reservoirs, validate_graph_and_reservoirs,
    outcome_counts_from_reservoirs, counts_to_pi, borda_scores, top_borda_table,
    ReplayEnvironment, run_ends_shortlist, run_static_sampler, run_thompson_shortlist_baseline,
    summarize_runs
)
from run_complete_arena_experiment import (
    run_borda_lucb_shortlist, run_top1_ends_plus_padding, run_empirical_top3_frequency,
    aggregate_budget, aggregate_fc, run_fixed_confidence_single
)


## 1. Full experiment configuration

In [ ]:

# ========== Real data path ==========
# Put your downloaded parquet file(s) here. Examples:
# Windows: r"D:/Datasets/Arena140K/*.parquet"
# macOS/Linux: "/Users/yourname/Datasets/Arena140K/*.parquet"
# If this is None, the notebook will load from Hugging Face via datasets.load_dataset.
LOCAL_PARQUET_GLOB = None
DATASET_ID = "lmarena-ai/arena-human-preference-140k"
CACHE_DIR = None

# Optional slice filters. Leave as None for the full logged Arena sample.
LANGUAGE = None       # e.g. "English" or "en" depending on actual dataset label values
IS_CODE = None        # True / False / None
START_TIMESTAMP = None
END_TIMESTAMP = None

# ========== Main non-toy experiment settings ==========
K_TARGET = 20
MIN_PAIR_SUPPORT = 20
CANDIDATE_MODEL_POOL_SIZE = 53
REPLAY_FRAC = 0.60

SHORTLIST_SIZE = 3
LAMBDA_BB = 0.0       # both_bad gives zero credit
ALPHA = 0.5           # Dirichlet smoothing/posterior parameter

BUDGETS = [500, 1000, 2000, 5000, 10000, 20000]
SEEDS = list(range(20))

# Exact corrected outsider-scenario ENDS settings.
# exhaustive_candidates=True enumerates all C(20,3)=1140 shortlists but is much slower.
EXHAUSTIVE_CANDIDATES = False
TOP_POOL = 12
MAX_CANDIDATES = 80
PROJECTION_MAXITER = 160
PROGRESS_EVERY = 0

OUTDIR = Path("outputs/notebook_complete_k20_support20")
OUTDIR.mkdir(parents=True, exist_ok=True)

config = {k: v for k, v in list(globals().items()) if k.isupper() and k not in {"OUTDIR"}}
config["OUTDIR"] = str(OUTDIR)
print(json.dumps(config, indent=2, ensure_ascii=False, default=str))


## 2. Load the real Arena140K data

In [ ]:

df_raw = load_arena_dataframe(
    dataset_id=DATASET_ID,
    cache_dir=CACHE_DIR,
    local_parquet_glob=LOCAL_PARQUET_GLOB,
)
df = filter_arena_dataframe(
    df_raw,
    language=LANGUAGE,
    is_code=IS_CODE,
    start_timestamp=START_TIMESTAMP,
    end_timestamp=END_TIMESTAMP,
)
print({
    "raw_rows": len(df_raw),
    "filtered_rows": len(df),
    "unique_models": int(pd.concat([df["model_a"], df["model_b"]]).nunique()),
    "winner_values": sorted(df["winner"].unique().tolist()),
})
assert len(df) > 0
assert {"model_a", "model_b", "winner"}.issubset(df.columns)
df.head()


## 3. Select K=20 models and build the eligible comparison graph

In [ ]:

selected_models, model_counts, pair_counts = greedy_select_models(
    df,
    K=K_TARGET,
    min_pair_support=MIN_PAIR_SUPPORT,
    candidate_pool_size=CANDIDATE_MODEL_POOL_SIZE,
    seed=20260704,
)
graph = build_graph(selected_models, pair_counts, min_pair_support=MIN_PAIR_SUPPORT)

graph_info = {
    "K": graph.K,
    "E": graph.E,
    "complete_E": graph.K * (graph.K - 1) // 2,
    "density": graph.E / (graph.K * (graph.K - 1) / 2),
    "degree_min": int(graph.degrees.min()),
    "degree_median": float(np.median(graph.degrees)),
    "degree_max": int(graph.degrees.max()),
    "pair_support_min": int(graph.pair_support.min()),
    "pair_support_median": float(np.median(graph.pair_support)),
    "pair_support_max": int(graph.pair_support.max()),
}
print(json.dumps(graph_info, indent=2))
assert graph.K == K_TARGET
assert graph.E > 0
assert graph.degrees.min() > 0

selected_models_df = pd.DataFrame({
    "model_idx": range(graph.K),
    "model": graph.models,
    "total_battles_in_slice": [int(model_counts.get(m, 0)) for m in graph.models],
    "degree_in_G": graph.degrees,
})
selected_models_df.to_csv(OUTDIR / "selected_models.csv", index=False)
selected_models_df


In [ ]:

eligible_edges_df = pd.DataFrame({
    "edge_idx": range(graph.E),
    "i": [e[0] for e in graph.edges],
    "j": [e[1] for e in graph.edges],
    "model_i": [graph.models[e[0]] for e in graph.edges],
    "model_j": [graph.models[e[1]] for e in graph.edges],
    "support": graph.pair_support,
})
eligible_edges_df.to_csv(OUTDIR / "eligible_edges.csv", index=False)
eligible_edges_df.head(20)


## 4. Build pair reservoirs and 60/40 replay-oracle split

In [ ]:

reservoirs = build_pair_reservoirs(df, graph)
replay_reservoirs, oracle_reservoirs = split_reservoirs(
    reservoirs,
    replay_frac=REPLAY_FRAC,
    seed=20260705,
)
validation = validate_graph_and_reservoirs(graph, replay_reservoirs, oracle_reservoirs)
print(json.dumps(validation, indent=2))
assert validation["empty_replay_edges"] == 0
assert validation["empty_oracle_edges"] == 0

reservoir_summary = pd.DataFrame({
    "edge_idx": range(graph.E),
    "model_i": [graph.models[i] for i, j in graph.edges],
    "model_j": [graph.models[j] for i, j in graph.edges],
    "replay_n": [len(replay_reservoirs[e]) for e in range(graph.E)],
    "oracle_n": [len(oracle_reservoirs[e]) for e in range(graph.E)],
})
reservoir_summary.to_csv(OUTDIR / "reservoir_summary.csv", index=False)
reservoir_summary.describe()


## 5. Oracle Borda ranking from the held-out oracle pool

In [ ]:

oracle_counts = outcome_counts_from_reservoirs(oracle_reservoirs, E=graph.E)
oracle_pi = counts_to_pi(oracle_counts, alpha=ALPHA)
oracle_borda = borda_scores(oracle_pi, graph, lambda_bb=LAMBDA_BB)
oracle_best_idx = int(np.argmax(oracle_borda))
oracle_table = top_borda_table(oracle_pi, graph, lambda_bb=LAMBDA_BB, top=graph.K)
oracle_table.to_csv(OUTDIR / "oracle_borda_ranking.csv", index=False)
print({
    "oracle_best_idx": oracle_best_idx,
    "oracle_best_model": graph.models[oracle_best_idx],
    "oracle_best_borda": float(oracle_borda[oracle_best_idx]),
})
oracle_table


## 6. One complete ENDS-Shortlist run at a real budget

In [ ]:

# This is not a toy run: K=20, real replay reservoirs, corrected exact outsider-scenario projection.
# Choose a budget from BUDGETS. Larger budgets take longer.
ONE_BUDGET = 1000
ONE_SEED = 0

one_ends = run_ends_shortlist(
    ReplayEnvironment(replay_reservoirs, graph, seed=9001, replace=True),
    graph,
    oracle_best_idx=oracle_best_idx,
    budget=max(ONE_BUDGET, graph.E + 1),
    m=SHORTLIST_SIZE,
    alpha=ALPHA,
    lambda_bb=LAMBDA_BB,
    seed=9002,
    exhaustive_candidates=EXHAUSTIVE_CANDIDATES,
    top_pool=TOP_POOL,
    max_candidates=MAX_CANDIDATES,
    progress_every=50,
    projection_maxiter=PROJECTION_MAXITER,
    nomination_mode="exact",
    detect_mode="exact",
)

one_summary = summarize_runs([one_ends], graph, oracle_borda=oracle_borda)
one_summary.to_csv(OUTDIR / "one_ends_run_summary.csv", index=False)
print(one_summary.T)
one_ends.trace.tail()


## 7. Full fixed-budget experiment over all algorithms, budgets, and seeds

In [ ]:

RUN_FULL_FIXED_BUDGET = True

if RUN_FULL_FIXED_BUDGET:
    all_summaries = []
    all_traces = []
    for budget in BUDGETS:
        budget_eff = max(int(budget), graph.E + 1)
        for seed in SEEDS:
            print(f"budget={budget_eff}, seed={seed}")
            results = []
            base = 20260704 + 100000 * seed + 1000 * budget_eff

            r_ends = run_ends_shortlist(
                ReplayEnvironment(replay_reservoirs, graph, seed=base+1, replace=True),
                graph, oracle_best_idx=oracle_best_idx, budget=budget_eff,
                m=SHORTLIST_SIZE, alpha=ALPHA, lambda_bb=LAMBDA_BB, seed=base+2,
                exhaustive_candidates=EXHAUSTIVE_CANDIDATES, top_pool=TOP_POOL,
                max_candidates=MAX_CANDIDATES, progress_every=PROGRESS_EVERY,
                projection_maxiter=PROJECTION_MAXITER, nomination_mode="exact", detect_mode="exact",
            )
            results.append(r_ends)
            if len(r_ends.trace):
                t = r_ends.trace.copy(); t.insert(0, "seed", seed); t.insert(0, "budget_requested", budget_eff); all_traces.append(t)

            results.append(run_static_sampler(
                ReplayEnvironment(replay_reservoirs, graph, seed=base+11, replace=True),
                graph, oracle_best_idx=oracle_best_idx, budget=budget_eff,
                m=SHORTLIST_SIZE, lambda_bb=LAMBDA_BB, seed=base+12,
                sampler="uniform", output_rule="empirical_top_m",
            ))
            results.append(run_static_sampler(
                ReplayEnvironment(replay_reservoirs, graph, seed=base+21, replace=True),
                graph, oracle_best_idx=oracle_best_idx, budget=budget_eff,
                m=SHORTLIST_SIZE, lambda_bb=LAMBDA_BB, seed=base+22,
                sampler="arena_frequency", pair_probs=graph.pair_support,
                output_rule="empirical_top_m",
            ))
            results.append(run_thompson_shortlist_baseline(
                ReplayEnvironment(replay_reservoirs, graph, seed=base+31, replace=True),
                graph, oracle_best_idx=oracle_best_idx, budget=budget_eff,
                m=SHORTLIST_SIZE, alpha=ALPHA, lambda_bb=LAMBDA_BB, seed=base+32,
            ))
            results.append(run_borda_lucb_shortlist(
                ReplayEnvironment(replay_reservoirs, graph, seed=base+41, replace=True),
                graph, oracle_best_idx=oracle_best_idx, budget=budget_eff,
                m=SHORTLIST_SIZE, alpha=ALPHA, lambda_bb=LAMBDA_BB, seed=base+42,
            ))
            results.append(run_top1_ends_plus_padding(
                ReplayEnvironment(replay_reservoirs, graph, seed=base+51, replace=True),
                graph, oracle_best_idx=oracle_best_idx, budget=budget_eff,
                m=SHORTLIST_SIZE, alpha=ALPHA, lambda_bb=LAMBDA_BB, seed=base+52,
                exhaustive_candidates=EXHAUSTIVE_CANDIDATES, top_pool=TOP_POOL,
                max_candidates=MAX_CANDIDATES, projection_maxiter=PROJECTION_MAXITER,
                progress_every=PROGRESS_EVERY,
            ))
            results.append(run_empirical_top3_frequency(
                ReplayEnvironment(replay_reservoirs, graph, seed=base+61, replace=True),
                graph, oracle_best_idx=oracle_best_idx, budget=budget_eff,
                m=SHORTLIST_SIZE, lambda_bb=LAMBDA_BB, seed=base+62,
                pair_probs=graph.pair_support,
            ))

            s = summarize_runs(results, graph, oracle_borda=oracle_borda)
            s.insert(0, "seed", seed)
            s.insert(0, "budget_requested", budget_eff)
            all_summaries.append(s)

    fixed_budget_df = pd.concat(all_summaries, ignore_index=True)
    fixed_budget_df.to_csv(OUTDIR / "fixed_budget_run_level.csv", index=False)
    fixed_budget_agg = aggregate_budget(fixed_budget_df)
    fixed_budget_agg.to_csv(OUTDIR / "fixed_budget_aggregate.csv", index=False)
    if all_traces:
        pd.concat(all_traces, ignore_index=True).to_csv(OUTDIR / "ends_trace.csv", index=False)

fixed_budget_agg if RUN_FULL_FIXED_BUDGET else None


## 8. Fixed-confidence calibration and held-out evaluation

In [ ]:

RUN_FIXED_CONFIDENCE = False  # Set True for the full threshold-calibration experiment.
DELTAS = [0.10, 0.05, 0.01]
THRESHOLD_CANDIDATES = [1.0, 2.0, 4.0, 8.0]
VALIDATION_SEEDS = list(range(20))
TEST_SEEDS = list(range(20, 40))
FC_MAX_BUDGET = 20000
FC_CHECK_EVERY = 50

# Build a small args-like object required by run_fixed_confidence_single.
class Args: pass
args = Args()
args.seed_base = 20260704
args.shortlist_size = SHORTLIST_SIZE
args.alpha = ALPHA
args.lambda_bb = LAMBDA_BB
args.exhaustive_candidates = EXHAUSTIVE_CANDIDATES
args.top_pool = TOP_POOL
args.max_candidates = MAX_CANDIDATES
args.fc_check_every = FC_CHECK_EVERY
args.projection_maxiter = PROJECTION_MAXITER

if RUN_FIXED_CONFIDENCE:
    val_rows = []
    for delta in DELTAS:
        for c in THRESHOLD_CANDIDATES:
            for seed in VALIDATION_SEEDS:
                print(f"validation delta={delta}, c={c}, seed={seed}")
                val_rows.append(run_fixed_confidence_single(
                    replay_reservoirs, graph, oracle_best_idx, oracle_borda,
                    seed=seed, delta=delta, c=c, max_budget=FC_MAX_BUDGET, args=args,
                ))
    fc_val = pd.concat(val_rows, ignore_index=True)
    fc_val.to_csv(OUTDIR / "fixed_confidence_validation.csv", index=False)
    fc_val_agg = aggregate_fc(fc_val)
    fc_val_agg.to_csv(OUTDIR / "fixed_confidence_validation_aggregate.csv", index=False)

    chosen = {}
    for delta in DELTAS:
        sub = fc_val_agg[fc_val_agg["delta"] == delta].sort_values("threshold_c")
        ok = sub[sub["empirical_failure"] <= delta]
        chosen[delta] = float(ok.iloc[0]["threshold_c"] if len(ok) else sub.iloc[-1]["threshold_c"])
    print("chosen c:", chosen)

    test_rows = []
    for delta, c in chosen.items():
        for seed in TEST_SEEDS:
            print(f"test delta={delta}, c={c}, seed={seed}")
            test_rows.append(run_fixed_confidence_single(
                replay_reservoirs, graph, oracle_best_idx, oracle_borda,
                seed=seed, delta=delta, c=c, max_budget=FC_MAX_BUDGET, args=args,
            ))
    fc_test = pd.concat(test_rows, ignore_index=True)
    fc_test.to_csv(OUTDIR / "fixed_confidence_test.csv", index=False)
    fc_test_agg = aggregate_fc(fc_test)
    fc_test_agg.to_csv(OUTDIR / "fixed_confidence_aggregate.csv", index=False)

fc_test_agg if RUN_FIXED_CONFIDENCE else "Fixed-confidence experiment is disabled. Set RUN_FIXED_CONFIDENCE=True to run it."


## 9. Plot and inspect final results

In [ ]:

import matplotlib.pyplot as plt

agg_path = OUTDIR / "fixed_budget_aggregate.csv"
if agg_path.exists():
    agg = pd.read_csv(agg_path)
    display(agg)
    plt.figure(figsize=(9, 5))
    for alg, sub in agg.groupby("algorithm"):
        sub = sub.sort_values("budget_requested")
        plt.plot(sub["budget_requested"], sub["success_rate"], marker="o", label=alg)
    plt.xscale("log")
    plt.xlabel("budget")
    plt.ylabel("success rate: oracle best in shortlist")
    plt.title("Arena140K ENDS-Shortlist fixed-budget success")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTDIR / "fixed_budget_success_curve.png", dpi=200)
    plt.show()
else:
    print("No fixed_budget_aggregate.csv yet.")


## 10. Equivalent command-line full run

In [ ]:

print(f"""
python run_complete_arena_experiment.py \
  --local-parquet-glob {repr(LOCAL_PARQUET_GLOB)} \
  --outdir {str(OUTDIR)!r} \
  --k-target {K_TARGET} \
  --min-pair-support {MIN_PAIR_SUPPORT} \
  --budgets {' '.join(map(str, BUDGETS))} \
  --seeds {' '.join(map(str, SEEDS))} \
  --run-fixed-budget \
  --algorithms ends uniform arena_frequency thompson lucb top1_padding empirical_top3 \
  --top-pool {TOP_POOL} \
  --max-candidates {MAX_CANDIDATES} \
  --projection-maxiter {PROJECTION_MAXITER}
""")
